<a href="https://colab.research.google.com/github/renatobdo/doutorado/blob/main/02_normalizacao_cid_sim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Título e objetivo

# Normalização e enriquecimento dos códigos CID-10 do SIM

Este notebook realiza a normalização dos códigos CID-10 presentes nas causas
dos óbitos cuja causa básica foi dengue.

As principais etapas são:

1. Carregamento dos dados extraídos do SIM;
2. Seleção das colunas relacionadas às causas do óbito;
3. Transformação das colunas de causas para formato longitudinal;
4. Separação de múltiplos códigos registrados no mesmo campo;
5. Normalização dos códigos CID-10;
6. Associação dos códigos às descrições da CID-10;
7. Validação dos códigos não encontrados;
8. Geração e salvamento da base enriquecida.

O resultado deste notebook será utilizado nas análises de causas e condições
mencionadas nos óbitos por dengue.

2 - Importação das bibliotecas

In [1]:
import os
import re
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


3. Definição dos caminhos

In [ ]:
import os
import pandas as pd

BASE_DIR = (
    "/content/drive/MyDrive/Doutorado/"
    "arbovirus_rag"
)

CAMINHO_ENTRADA = os.path.join(
    BASE_DIR,
    "data_processed",
    "sim_obitos_dengue_2023_reduzido.csv"
)

CAMINHO_CATEGORIAS = os.path.join(
    BASE_DIR,
    "data_docs",
    "CID10CSV",
    "CID-10-CATEGORIAS.CSV"
)

CAMINHO_SUBCATEGORIAS = os.path.join(
    BASE_DIR,
    "data_docs",
    "CID10CSV",
    "CID-10-SUBCATEGORIAS.CSV"
)

CAMINHO_SAIDA_CAUSAS = os.path.join(
    BASE_DIR,
    "data_processed",
    "sim_obitos_dengue_2023_causas_cid10.csv"
)

CAMINHO_SAIDA_FREQUENCIAS = os.path.join(
    BASE_DIR,
    "data_processed",
    "sim_obitos_dengue_2023_frequencia_cid10.csv"
)

4. Carregamento dos dados do notebook 01

In [ ]:
if not os.path.exists(CAMINHO_ENTRADA):
    raise FileNotFoundError(
        f"Arquivo de entrada não encontrado: {CAMINHO_ENTRADA}"
    )

df_sim = pd.read_csv(CAMINHO_ENTRADA)

print("Base carregada com sucesso.")
print(f"Registros: {len(df_sim):,}")
print(f"Colunas: {df_sim.shape[1]}")

(1322, 88)


,ID_REGISTRO,ORIGEM,TIPOBITO,DTOBITO,HORAOBITO,NATURAL,CODMUNNATU,DTNASC,IDADE,SEXO,...,FONTES,TPRESGINFO,TPNIVELINV,NUDIASINF,DTCADINF,MORTEPARTO,DTCONCASO,FONTESINF,ALTCAUSA,CONTADOR
0,1,1,2,02012023,0550,829,290140,20011957,465,2,...,,,,,,,,,,4783
1,2,1,2,04012023,1352,829,292210,15051939,483,1,...,,,,,,,,,,15755
2,3,1,2,05012023,2037,832,320320,28121967,455,2,...,,,,,,,,,,19217
3,4,1,2,06012023,1940,824,240380,22021922,500,1,...,,,,,,,,,,21110
4,5,1,2,09012023,0756,832,320140,07091945,477,2,...,,,,,,,,,,34522


Confira as colunas:

In [ ]:
df_sim.columns.tolist()

,ID_REGISTRO,CAMPO_CAUSA,VALOR_ORIGINAL
0,1,CAUSABAS,A91
1,2,CAUSABAS,A91
2,3,CAUSABAS,A91
3,4,CAUSABAS,A91
4,5,CAUSABAS,A90


5. Criação de um identificador para cada óbito
O identificador deve ser criado antes de transformar as causas em linhas:

In [ ]:
df_sim = df_sim.reset_index(drop=True).copy()

if "ID_REGISTRO" not in df_sim.columns:
    df_sim.insert(
        0,
        "ID_REGISTRO",
        df_sim.index + 1
    )

Validação:

In [ ]:
assert df_sim["ID_REGISTRO"].is_unique
assert df_sim["ID_REGISTRO"].notna().all()

df_sim[["ID_REGISTRO", "CAUSABAS"]].head()

,ID_REGISTRO,CAMPO_CAUSA,VALOR_ORIGINAL,CODIGO_SIM
0,1,CAUSABAS,A91,A91
1,2,CAUSABAS,A91,A91
2,3,CAUSABAS,A91,A91
3,4,CAUSABAS,A91,A91
4,5,CAUSABAS,A90,A90
5,6,CAUSABAS,A91,A91
6,7,CAUSABAS,A90,A90
7,8,CAUSABAS,A90,A90
8,9,CAUSABAS,A91,A91
9,10,CAUSABAS,A91,A91


É importante deixar explícito que esse identificador é interno:

> `ID_REGISTRO` é um identificador criado para processamento no notebook.
> Ele não corresponde a um identificador oficial disponibilizado pelo SIM.

In [ ]:
# Código completo extraído do SIM
df_causas_separadas["CODIGO_COMPLETO"] = (
    df_causas_separadas["CODIGO_SIM"]
)

# Categoria CID-10: sempre os três primeiros caracteres
df_causas_separadas["CODIGO_CATEGORIA"] = (
    df_causas_separadas["CODIGO_SIM"]
    .str[:3]
)

# Código candidato à subcategoria
df_causas_separadas["CODIGO_SUBCATEGORIA"] = (
    df_causas_separadas["CODIGO_SIM"]
)

6. Seleção das colunas de causas

In [ ]:
COLUNAS_CAUSAS = [
    "CAUSABAS",
    "LINHAA",
    "LINHAB",
    "LINHAC",
    "LINHAD",
    "LINHAII"
]

Valide se todas estão disponíveis:

In [ ]:
colunas_ausentes = [
    coluna
    for coluna in COLUNAS_CAUSAS
    if coluna not in df_sim.columns
]

if colunas_ausentes:
    raise KeyError(
        f"Colunas de causas ausentes: {colunas_ausentes}"
    )

Você também pode registrar o papel dos campos:

In [ ]:
PAPEIS_CAUSAS = {
    "CAUSABAS": "Causa básica",
    "LINHAA": "Linha A da Parte I",
    "LINHAB": "Linha B da Parte I",
    "LINHAC": "Linha C da Parte I",
    "LINHAD": "Linha D da Parte I",
    "LINHAII": "Outras condições significativas — Parte II"
}

Arquivo lido com sep=';' e encoding='latin-1'.
Arquivo lido com sep=';' e encoding='latin-1'.
Categorias:
['CAT', 'CLASSIF', 'DESCRICAO', 'DESCRABREV', 'REFER', 'EXCLUIDOS', 'Unnamed: 6']


,CAT,CLASSIF,DESCRICAO,DESCRABREV,REFER,EXCLUIDOS,Unnamed: 6
0,A00,<NA>,Cólera,A00 Colera,<NA>,<NA>,<NA>
1,A01,<NA>,Febres tifóide e paratifóide,A01 Febres tifoide e paratifoide,<NA>,<NA>,<NA>
2,A02,<NA>,Outras infecções por Salmonella,A02 Outr infecc p/Salmonella,<NA>,<NA>,<NA>
3,A03,<NA>,Shiguelose,A03 Shiguelose,<NA>,<NA>,<NA>
4,A04,<NA>,Outras infecções intestinais bacterianas,A04 Outr infecc intestinais bacter,<NA>,<NA>,<NA>


Subcategorias:
['SUBCAT', 'CLASSIF', 'RESTRSEXO', 'CAUSAOBITO', 'DESCRICAO', 'DESCRABREV', 'REFER', 'EXCLUIDOS', 'Unnamed: 8']


,SUBCAT,CLASSIF,RESTRSEXO,CAUSAOBITO,DESCRICAO,DESCRABREV,REFER,EXCLUIDOS,Unnamed: 8
0,A000,<NA>,<NA>,<NA>,"Cólera devida a Vibrio cholerae 01, biótipo ch...",A00.0 Colera dev Vibrio cholerae 01 biot cholerae,<NA>,<NA>,<NA>
1,A001,<NA>,<NA>,<NA>,"Cólera devida a Vibrio cholerae 01, biótipo El...",A00.1 Colera dev Vibrio cholerae 01 biot El Tor,<NA>,<NA>,<NA>
2,A009,<NA>,<NA>,<NA>,Cólera não especificada,A00.9 Colera NE,<NA>,<NA>,<NA>
3,A010,<NA>,<NA>,<NA>,Febre tifóide,A01.0 Febre tifoide,<NA>,<NA>,<NA>
4,A011,<NA>,<NA>,<NA>,Febre paratifóide A,A01.1 Febre paratifoide A,<NA>,<NA>,<NA>


7. Transformação para formato longitudinal

No formato original, cada tipo de causa ocupa uma coluna. Para análise, é mais adequado criar uma linha para cada campo:

In [ ]:
df_causas_long = df_sim.melt(
    id_vars=["ID_REGISTRO"],
    value_vars=COLUNAS_CAUSAS,
    var_name="CAMPO_CAUSA",
    value_name="VALOR_ORIGINAL"
)

Adicione a descrição do campo:

In [ ]:
df_causas_long["PAPEL_CAUSA"] = (
    df_causas_long["CAMPO_CAUSA"]
    .map(PAPEIS_CAUSAS)
)

Remova valores vazios:

In [ ]:
df_causas_long["VALOR_ORIGINAL"] = (
    df_causas_long["VALOR_ORIGINAL"]
    .astype("string")
    .str.strip()
)

df_causas_long = df_causas_long[
    df_causas_long["VALOR_ORIGINAL"].notna()
    &
    df_causas_long["VALOR_ORIGINAL"].ne("")
].copy()

Visualização:

In [ ]:
df_causas_long.head(20)

8. Separação automática dos códigos CID-10

Esta é uma das partes centrais do notebook.

Valores como:

*I10X*E149

devem gerar duas linhas:

I10X
E149

Use a expressão regular:

In [ ]:
PADRAO_CID = r"[A-Z][0-9]{2}[A-Z0-9]?"

Extração:

In [ ]:
df_causas_long["CODIGOS_EXTRAIDOS"] = (
    df_causas_long["VALOR_ORIGINAL"]
    .str.upper()
    .str.findall(PADRAO_CID)
)

Explosão da lista de códigos:

In [ ]:
df_causas_separadas = (
    df_causas_long
    .explode("CODIGOS_EXTRAIDOS")
    .rename(
        columns={
            "CODIGOS_EXTRAIDOS": "CODIGO_SIM"
        }
    )
    .dropna(subset=["CODIGO_SIM"])
    .copy()
)

Normalize:

In [ ]:
df_causas_separadas["CODIGO_SIM"] = (
    df_causas_separadas["CODIGO_SIM"]
    .astype("string")
    .str.upper()
    .str.strip()
    .str.replace(".", "", regex=False)
)

Exemplo de validação:

In [ ]:
df_causas_separadas[
    df_causas_separadas["VALOR_ORIGINAL"]
    .str.contains(r"\*.*\*", regex=True, na=False)
][
    [
        "ID_REGISTRO",
        "CAMPO_CAUSA",
        "VALOR_ORIGINAL",
        "CODIGO_SIM"
    ]
].head(20)